In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import pipelines as dp

In [0]:
df=spark.read.format("delta")\
    .load("/Volumes/workspace/bronze/bronzevolume/bookings/data/")
display(df)

In [0]:
df=df.withColumn("amount",col("amount").cast(DoubleType()))\
        .withColumn("modifiedDate",current_timestamp())\
        .withColumn("booking_date",to_date(col("booking_date")))\
        .drop("_rescued_data")
display(df)

In [0]:
@dp.table(
    name="stage_bookings"
)
def stage_bookings():

    df=spark.readStream.format("delta")
        .load("/Volumes/workspace/bronze/bronzevolume/bookings/data")
    return df


In [0]:
@dp.view(
    name="trans_bookings"
)
def trans_bookings():

    df=spark.readStream.table("stage_bookings")
    df=df.withColumn("amount",col("amount").cast(DoubleType()))\
        .withColumn("modifiedDate",current_timestamp())\
        .withColumn("booking_date",to_date(col("booking_date")))\
        .drop("_rescued_data")
    return df


In [0]:
rules={
    "rule1":"booking_id IS NOT NULL",
    "rule2":"passenger_id IS NOT NULL",
}
 

In [0]:
@dp.table(
    name="silver_bookings"
)
@dp.expect_all(rules)
def silver_bookings():

    df=spark.readStream.table("trans_bookings")
    return df